In [1]:
import queue

from ib_insync import Stock, MarketOrder, IB, util

from systems import IBKRConnection, HistoricPandasDataHandler, IBKRLiveDataHandler, TradingEngine
from events import MarketEvent, SignalEvent, OrderEvent, FillEvent
from strategy import MovingAverageStrategy, MachineLearningStrategy
from portfolio_manager import PortfolioManager
from execution import ExecutionHandler, SimulatedExecutionHandler

import nest_asyncio
nest_asyncio.apply()

In [2]:
# --- CHOOSE YOUR MODE ---
MODE = "BACKTEST" # Change to "LIVE" when ready
symbol = 'AAPL'
events_queue = queue.Queue()

# 1. Connect to IBKR
ib_conn = IBKRConnection(live_trading=False) # live_trading=False uses paper port 7497
ib_conn.connect()

Paper trading mode enabled.
Connecting to IBKR at 127.0.0.1:7497 with client ID 1
Connected to IBKR at 127.0.0.1:7497 with client ID 1


<IB connected to 127.0.0.1:7497 clientId=1>

In [3]:

contract = Stock(symbol, 'SMART', 'USD')
if MODE == "BACKTEST":
    
    dummy_df = ib_conn.get_data(contract, durationStr="60 D", barSizeSetting="1 hour")
    data_handler = HistoricPandasDataHandler(events_queue, dummy_df, symbol)
    strategy = MovingAverageStrategy(events_queue, data_handler, symbol, fast_period=10, slow_period=30)
    portfolio = PortfolioManager(events_queue, data_handler, initial_capital=100000.0)
    execution = SimulatedExecutionHandler(events_queue, data_handler)

    # 3. Run Engine
    engine = TradingEngine(data_handler, strategy, portfolio, execution, events_queue)
    engine.run()

Starting Trading Engine Loop...
[29] SIGNAL: Fast MA (273.95) < Slow MA (275.74). Going SHORT/FLAT.
[63] SIGNAL: Fast MA (271.85) > Slow MA (271.73). Going LONG.
[PORTFOLIO] Approved LONG signal. Generated BUY order for 10 AAPL.
[EXECUTION - SIM] FILLED BUY 10 AAPL @ $274.19
[PORTFOLIO] Fill received. New Cash Balance: $97257.10 | Holdings: {'AAPL': 10}
[85] SIGNAL: Fast MA (273.29) < Slow MA (273.37). Going SHORT/FLAT.
[PORTFOLIO] Approved SHORT/EXIT signal. Generated SELL order to close 10 AAPL.
[EXECUTION - SIM] FILLED SELL 10 AAPL @ $273.24
[PORTFOLIO] Fill received. New Cash Balance: $99988.50 | Holdings: {'AAPL': 0}
[142] SIGNAL: Fast MA (259.84) > Slow MA (259.73). Going LONG.
[PORTFOLIO] Approved LONG signal. Generated BUY order for 10 AAPL.
[EXECUTION - SIM] FILLED BUY 10 AAPL @ $261.13
[PORTFOLIO] Fill received. New Cash Balance: $97376.20 | Holdings: {'AAPL': 10}
[156] SIGNAL: Fast MA (258.94) < Slow MA (259.15). Going SHORT/FLAT.
[PORTFOLIO] Approved SHORT/EXIT signal. Gene

In [ ]:
ib_conn.disconnect()

Disconnecting from IBKR ...
Disconnected from IBKR.


: 